# Patient Folder ZIP Processing
This notebook counts patient folders, validates ZIP files, extracts each ZIP, and then moves original ZIP files into an archive folder so they are preserved (not deleted).

## 1) Import Libraries and Set Root Directory

In [19]:
import os
import zipfile
import shutil
import gzip

## 2) Discover and Count Patient Folders

In [20]:
root_dir = r"E:\validation_data"   # change this
zip_store = os.path.join(root_dir, "compressed_files")

os.makedirs(zip_store, exist_ok=True)

## 3) Validate ZIP Files in Each Patient Folder

In [21]:
def safe_move(src, dest_dir):
    name = os.path.basename(src)
    dest = os.path.join(dest_dir, name)

    counter = 1
    while os.path.exists(dest):
        base, ext = os.path.splitext(name)
        dest = os.path.join(dest_dir, f"{base}_{counter}{ext}")
        counter += 1

    shutil.move(src, dest)


def decompress_nii_gz(file_path):
    output_path = file_path.replace(".nii.gz", ".nii")

    with gzip.open(file_path, 'rb') as f_in:
        with open(output_path, 'wb') as f_out:
            shutil.copyfileobj(f_in, f_out)

    os.remove(file_path)  # remove .nii.gz after extract

## 4) Extract ZIP Files per Patient

In [22]:
expected_modalities = ["t1c", "t1n", "t2f", "t2w"]

## 5) Create Archive Folder and Move ZIP Files

In [23]:
patients = [
    f for f in os.listdir(root_dir)
    if os.path.isdir(os.path.join(root_dir, f))
    and f != "compressed_files"
]

for patient in patients:
    patient_path = os.path.join(root_dir, patient)
    print(f"\nProcessing: {patient}")

    # 🔸 Step 1: Unzip all zip files
    for file in os.listdir(patient_path):
        if file.endswith(".zip"):
            zip_path = os.path.join(patient_path, file)

            try:
                with zipfile.ZipFile(zip_path, 'r') as zip_ref:
                    zip_ref.extractall(patient_path)

                safe_move(zip_path, zip_store)
                print(f"  Unzipped & moved zip: {file}")

            except Exception as e:
                print(f"  Error unzip: {file} -> {e}")

    # 🔸 Step 2: Walk all files
    for root, dirs, files in os.walk(patient_path):
        for file in files:
            full_path = os.path.join(root, file)

            # Convert .nii.gz → .nii
            if file.endswith(".nii.gz"):
                decompress_nii_gz(full_path)
                full_path = full_path.replace(".nii.gz", ".nii")

            # Move .nii to root patient folder
            if full_path.endswith(".nii"):
                dest = os.path.join(patient_path, os.path.basename(full_path))
                if full_path != dest:
                    shutil.move(full_path, dest)

    # 🔸 Step 3: Remove all subfolders
    for item in os.listdir(patient_path):
        item_path = os.path.join(patient_path, item)
        if os.path.isdir(item_path):
            shutil.rmtree(item_path)

    # 🔸 Step 4: Validation
    nii_files = [f for f in os.listdir(patient_path) if f.endswith(".nii")]

    found_modalities = []
    for f in nii_files:
        for mod in expected_modalities:
            if f.endswith(f"-{mod}.nii"):
                found_modalities.append(mod)

    missing = list(set(expected_modalities) - set(found_modalities))

    # 🔸 Result Print
    print(f"  Found: {found_modalities}")

    if not missing and len(nii_files) == 4:
        print("  ✅ VALID (4 correct .nii files)")
    else:
        if missing:
            print(f"  ❌ Missing: {missing}")
        if len(nii_files) != 4:
            print(f"  ⚠ File count issue: {len(nii_files)} files found")


Processing: BraTS-GLI-02073-100
  Found: ['t1c', 't1n', 't2f', 't2w']
  ✅ VALID (4 correct .nii files)

Processing: BraTS-GLI-02073-101
  Found: ['t1c', 't1n', 't2f', 't2w']
  ✅ VALID (4 correct .nii files)

Processing: BraTS-GLI-02095-100
  Found: ['t1c', 't1n', 't2f', 't2w']
  ✅ VALID (4 correct .nii files)

Processing: BraTS-GLI-02095-101
  Found: ['t1c', 't1n', 't2f', 't2w']
  ✅ VALID (4 correct .nii files)

Processing: BraTS-GLI-02095-102
  Found: ['t1c', 't1n', 't2f', 't2w']
  ✅ VALID (4 correct .nii files)

Processing: BraTS-GLI-02097-100
  Found: ['t1c', 't1n', 't2f', 't2w']
  ✅ VALID (4 correct .nii files)

Processing: BraTS-GLI-02097-101
  Found: ['t1c', 't1n', 't2f', 't2w']
  ✅ VALID (4 correct .nii files)

Processing: BraTS-GLI-02109-100
  Found: ['t1c', 't1n', 't2f', 't2w']
  ✅ VALID (4 correct .nii files)

Processing: BraTS-GLI-02109-101
  Found: ['t1c', 't1n', 't2f', 't2w']
  ✅ VALID (4 correct .nii files)

Processing: BraTS-GLI-02109-102
  Found: ['t1c', 't1n', 't2f', 

## 6) Generate Processing Report

In [24]:
import os

# Path to dataset
root_dir = r"E:\validation_data"   # change this

expected_modalities = ["t1c", "t1n", "t2f", "t2w"]

patients = [
    f for f in os.listdir(root_dir)
    if os.path.isdir(os.path.join(root_dir, f))
]

for patient in patients:
    patient_path = os.path.join(root_dir, patient)

    # Get only .nii files
    nii_files = [f for f in os.listdir(patient_path) if f.endswith(".nii")]

    found_modalities = []

    # Detect modalities from filenames
    for f in nii_files:
        for mod in expected_modalities:
            if f.endswith(f"-{mod}.nii"):
                found_modalities.append(mod)

    found_set = set(found_modalities)
    expected_set = set(expected_modalities)

    missing = expected_set - found_set
    extra = found_set - expected_set

    print(f"\nPatient: {patient}")
    print(f"  Found: {sorted(found_set)}")

    if not missing and not extra and len(nii_files) == 4:
        print("  ✅ VALID")
    else:
        if missing:
            print(f"  ❌ Missing: {sorted(missing)}")
        if extra:
            print(f"  ⚠ Extra modalities: {sorted(extra)}")
        if len(nii_files) != 4:
            print(f"  ⚠ File count issue: {len(nii_files)} files")


Patient: BraTS-GLI-02073-100
  Found: ['t1c', 't1n', 't2f', 't2w']
  ✅ VALID

Patient: BraTS-GLI-02073-101
  Found: ['t1c', 't1n', 't2f', 't2w']
  ✅ VALID

Patient: BraTS-GLI-02095-100
  Found: ['t1c', 't1n', 't2f', 't2w']
  ✅ VALID

Patient: BraTS-GLI-02095-101
  Found: ['t1c', 't1n', 't2f', 't2w']
  ✅ VALID

Patient: BraTS-GLI-02095-102
  Found: ['t1c', 't1n', 't2f', 't2w']
  ✅ VALID

Patient: BraTS-GLI-02097-100
  Found: ['t1c', 't1n', 't2f', 't2w']
  ✅ VALID

Patient: BraTS-GLI-02097-101
  Found: ['t1c', 't1n', 't2f', 't2w']
  ✅ VALID

Patient: BraTS-GLI-02109-100
  Found: ['t1c', 't1n', 't2f', 't2w']
  ✅ VALID

Patient: BraTS-GLI-02109-101
  Found: ['t1c', 't1n', 't2f', 't2w']
  ✅ VALID

Patient: BraTS-GLI-02109-102
  Found: ['t1c', 't1n', 't2f', 't2w']
  ✅ VALID

Patient: BraTS-GLI-02109-103
  Found: ['t1c', 't1n', 't2f', 't2w']
  ✅ VALID

Patient: BraTS-GLI-02109-104
  Found: ['t1c', 't1n', 't2f', 't2w']
  ✅ VALID

Patient: BraTS-GLI-02113-100
  Found: ['t1c', 't1n', 't2f', 't2